# Coffee Standard J25 — YOLOv8s matched baseline, seed 42

Matched prior-study detector baseline on the source-level J25 development split. Same data, 50-epoch budget, and semantic-safe augmentation as SAFEAUG0/CWCF1. Locked test remains closed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import csv, importlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
ARM='V8S_MATCHED'; BRANCH='codex/j25-yolov8s-matched-baseline'
REPO=Path('/content/coffee-bean-detection'); WORK=Path('/content')
os.chdir(WORK)
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.96','gdown'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','--no-deps','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
import torch
if not torch.cuda.is_available(): raise RuntimeError('Aktifkan GPU Colab')
experiment_roots=[Path('/content/drive/MyDrive/Coffee_Bean_Detection/experiments'),*Path('/content/drive/.shortcut-targets-by-id').glob('*/Coffee_Bean_Detection/experiments')]
def find_file(folder, relative):
    matches=[root/folder/relative for root in experiment_roots if (root/folder/relative).is_file()]
    if not matches: raise FileNotFoundError(f'{folder}/{relative} tidak ditemukan di Drive')
    return matches[0]
SAFEAUG_RESULT=find_file('coffee-standard-j25-safe-augmentation-v1',Path('val_reports/SAFEAUG0_seed42_result.json'))
CWCF_RESULT=find_file('coffee-standard-j25-cwcf-v1',Path('val_reports/CWCF1_seed42_result.json'))
PROJECT=SAFEAUG_RESULT.parents[3]
print('SAFEAUG0:',SAFEAUG_RESULT); print('CWCF1:',CWCF_RESULT); print('PROJECT:',PROJECT)


In [ ]:
from coffee_detector.analysis.coffee_standard_j25_thesis_provenance import audit_j25_thesis_provenance
from coffee_detector.data.prepare_coffee_standard_j25_source_split import prepare_j25_source_split
ARCHIVE=WORK/'data_aug_11.zip'
if not ARCHIVE.is_file(): subprocess.run([sys.executable,'-m','gdown','https://drive.google.com/uc?id=1AofT7VbiNFM8ul-0vyCAKj7Rp4j5OX0f','-O',str(ARCHIVE)],check=True)
PROVENANCE=WORK/'coffee_standard_j25_thesis_provenance.json'
provenance=audit_j25_thesis_provenance(ARCHIVE,PROVENANCE)
if not provenance['decision'].startswith('PASS'): raise RuntimeError(f'Provenance gagal: {provenance["decision"]}')
DATA=WORK/'coffee-standard-j25-train-siblings-v2'
if DATA.exists(): shutil.rmtree(DATA)
contract=prepare_j25_source_split(ARCHIVE,DATA,seed=42,retain_train_siblings=True)
CONTRACT=DATA/'coffee_standard_j25_train_siblings_summary.json'
from ultralytics import YOLO
_=YOLO('yolov8s.pt'); PRETRAINED=REPO/'yolov8s.pt'
if not PRETRAINED.is_file(): raise FileNotFoundError(PRETRAINED)
OUT=PROJECT/'experiments/coffee-standard-j25-yolov8s-matched-v1'; OUT.mkdir(parents=True,exist_ok=True)
print('GPU:',torch.cuda.get_device_name(0),'| DATA:',contract['images'],'| PRETRAINED:',PRETRAINED,'| OUT:',OUT)


In [ ]:
from coffee_detector.experiments.run_coffee_standard_j25_yolov8s_matched import run_static_preflight
STATIC=OUT/'static_preflight.json'
static=run_static_preflight(PRETRAINED,STATIC,seed=42)
print('STATIC:',static['decision']); print('GATES:',static['gates']); print('PARAMS:',static['source_parameters']); print('SHA256:',static['pretrained_checkpoint_sha256'])
assert static['decision']=='PASS', 'STOP: preflight gagal; training tidak dijalankan.'


In [ ]:
LOG=OUT/f'{ARM}_seed42_run.log'
command=[sys.executable,'-u','-m','coffee_detector.experiments.run_coffee_standard_j25_yolov8s_matched','--data-root',str(DATA),'--development-contract',str(CONTRACT),'--provenance-summary',str(PROVENANCE),'--pretrained-checkpoint',str(PRETRAINED),'--output-root',str(OUT),'--seed','42','--device','0','--authorize-training']
print('START/RESUME:',ARM,'| log=',LOG,flush=True)
with LOG.open('a',encoding='utf-8') as stream: process=subprocess.Popen(command,cwd=REPO,stdout=stream,stderr=subprocess.STDOUT)
last=-1
while process.poll() is None:
    results=OUT/ARM/f'{ARM}_seed42'/'results.csv'
    epochs=sum(1 for _ in csv.DictReader(results.open(encoding='utf-8'))) if results.is_file() else 0
    if epochs!=last: print(f'{ARM}: {epochs}/50 epoch tercatat',flush=True); last=epochs
    time.sleep(120)
if process.returncode:
    print('\n'.join(LOG.read_text(errors='replace').splitlines()[-120:])); raise RuntimeError(f'{ARM} gagal: {process.returncode}')
RESULT=OUT/'val_reports'/f'{ARM}_seed42_result.json'
result=json.loads(RESULT.read_text())
print('METRICS:',result['metrics']); print('TARGET:',result['target_class'],result['target_class_map50_95']); print('TEST:',result['test_images_accessed'])


In [ ]:
from coffee_detector.experiments.run_coffee_standard_j25_yolov8s_matched import build_comparison
SUMMARY=OUT/'yolov8s_matched_seed42_comparison.json'
comparison=build_comparison(RESULT,SAFEAUG_RESULT,CWCF_RESULT,SUMMARY)
print('VALUES:',comparison['values'])
print('V8S - SAFEAUG0:',comparison['v8s_minus_safeaug0'])
print('CWCF1 - V8S:',comparison['cwcf1_minus_v8s'])
print('TARGET:',comparison['target_values'])
print('DECISION:',comparison['decision'],'| TEST:',comparison['test_opened'])
print('NOTE: ordering ini deskriptif; bukan promotion/kill gate untuk CWCF1.')
